In [1]:
%load_ext autoreload
%autoreload 2

In [67]:
import numpy as np
import pandas as pd
import os
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
import config as cnf 

In [3]:
df_train = pd.read_csv(cnf.path['train_path'])
df_test = pd.read_csv(cnf.path['test_path'])
df_train

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,1456,60,RL,62.0,7917,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,8,2007,WD,Normal,175000
1456,1457,20,RL,85.0,13175,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,2,2010,WD,Normal,210000
1457,1458,70,RL,66.0,9042,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,GdPrv,Shed,2500,5,2010,WD,Normal,266500
1458,1459,20,RL,68.0,9717,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,142125


### EDA

In [4]:
df_train

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,1456,60,RL,62.0,7917,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,8,2007,WD,Normal,175000
1456,1457,20,RL,85.0,13175,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,2,2010,WD,Normal,210000
1457,1458,70,RL,66.0,9042,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,GdPrv,Shed,2500,5,2010,WD,Normal,266500
1458,1459,20,RL,68.0,9717,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,142125


In [9]:
df_train.select_dtypes(include=['int64', 'float64'], exclude=None).info()

<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 38 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   LotFrontage    1201 non-null   float64
 3   LotArea        1460 non-null   int64  
 4   OverallQual    1460 non-null   int64  
 5   OverallCond    1460 non-null   int64  
 6   YearBuilt      1460 non-null   int64  
 7   YearRemodAdd   1460 non-null   int64  
 8   MasVnrArea     1452 non-null   float64
 9   BsmtFinSF1     1460 non-null   int64  
 10  BsmtFinSF2     1460 non-null   int64  
 11  BsmtUnfSF      1460 non-null   int64  
 12  TotalBsmtSF    1460 non-null   int64  
 13  1stFlrSF       1460 non-null   int64  
 14  2ndFlrSF       1460 non-null   int64  
 15  LowQualFinSF   1460 non-null   int64  
 16  GrLivArea      1460 non-null   int64  
 17  BsmtFullBath   1460 non-null   int64  
 18  BsmtHalfBath   1460

In [11]:
df_train.select_dtypes(include=['int64', 'float64'], exclude=None)

,Id,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,...,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SalePrice
0,1,60,65.0,8450,7,5,2003,2003,196.0,706,...,0,61,0,0,0,0,0,2,2008,208500
1,2,20,80.0,9600,6,8,1976,1976,0.0,978,...,298,0,0,0,0,0,0,5,2007,181500
2,3,60,68.0,11250,7,5,2001,2002,162.0,486,...,0,42,0,0,0,0,0,9,2008,223500
3,4,70,60.0,9550,7,5,1915,1970,0.0,216,...,0,35,272,0,0,0,0,2,2006,140000
4,5,60,84.0,14260,8,5,2000,2000,350.0,655,...,192,84,0,0,0,0,0,12,2008,250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,1456,60,62.0,7917,6,5,1999,2000,0.0,0,...,0,40,0,0,0,0,0,8,2007,175000
1456,1457,20,85.0,13175,6,6,1978,1988,119.0,790,...,349,0,0,0,0,0,0,2,2010,210000
1457,1458,70,66.0,9042,7,9,1941,2006,0.0,275,...,0,60,0,0,0,0,2500,5,2010,266500
1458,1459,20,68.0,9717,5,6,1950,1996,0.0,49,...,366,0,112,0,0,0,0,4,2010,142125


In [10]:
df_train.select_dtypes(include=None, exclude=['int64', 'float64']).info()

<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 43 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   MSZoning       1460 non-null   str  
 1   Street         1460 non-null   str  
 2   Alley          91 non-null     str  
 3   LotShape       1460 non-null   str  
 4   LandContour    1460 non-null   str  
 5   Utilities      1460 non-null   str  
 6   LotConfig      1460 non-null   str  
 7   LandSlope      1460 non-null   str  
 8   Neighborhood   1460 non-null   str  
 9   Condition1     1460 non-null   str  
 10  Condition2     1460 non-null   str  
 11  BldgType       1460 non-null   str  
 12  HouseStyle     1460 non-null   str  
 13  RoofStyle      1460 non-null   str  
 14  RoofMatl       1460 non-null   str  
 15  Exterior1st    1460 non-null   str  
 16  Exterior2nd    1460 non-null   str  
 17  MasVnrType     588 non-null    str  
 18  ExterQual      1460 non-null   str  
 19  ExterCond      14

In [15]:
columns_with_na

['MSZoning',
 'LotFrontage',
 'Alley',
 'Utilities',
 'Exterior1st',
 'Exterior2nd',
 'MasVnrType',
 'MasVnrArea',
 'BsmtQual',
 'BsmtCond',
 'BsmtExposure',
 'BsmtFinType1',
 'BsmtFinSF1',
 'BsmtFinType2',
 'BsmtFinSF2',
 'BsmtUnfSF',
 'TotalBsmtSF',
 'BsmtFullBath',
 'BsmtHalfBath',
 'KitchenQual',
 'Functional',
 'FireplaceQu',
 'GarageType',
 'GarageYrBlt',
 'GarageFinish',
 'GarageCars',
 'GarageArea',
 'GarageQual',
 'GarageCond',
 'PoolQC',
 'Fence',
 'MiscFeature',
 'SaleType']

In [36]:
columns_with_na = df_train.columns[df_train.isna().any()].tolist()
filtered_df = df_train[columns_with_na]
filtered_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 19 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   LotFrontage   1201 non-null   float64
 1   Alley         1 non-null      str    
 2   MasVnrType    588 non-null    str    
 3   MasVnrArea    1452 non-null   float64
 4   BsmtQual      1423 non-null   str    
 5   BsmtCond      1423 non-null   str    
 6   BsmtExposure  1422 non-null   str    
 7   BsmtFinType1  1423 non-null   str    
 8   BsmtFinType2  1422 non-null   str    
 9   Electrical    1459 non-null   str    
 10  FireplaceQu   770 non-null    str    
 11  GarageType    1379 non-null   str    
 12  GarageYrBlt   1379 non-null   float64
 13  GarageFinish  1379 non-null   str    
 14  GarageQual    1379 non-null   str    
 15  GarageCond    1379 non-null   str    
 16  PoolQC        7 non-null      str    
 17  Fence         281 non-null    str    
 18  MiscFeature   54 non-null     str    
d

In [52]:
df_train['LotFrontage'].dtype

dtype('float64')

In [38]:
df_train['Alley']

0       Grvl
1        NaN
2        NaN
3        NaN
4        NaN
        ... 
1455     NaN
1456     NaN
1457     NaN
1458     NaN
1459     NaN
Name: Alley, Length: 1460, dtype: str

In [62]:
df_train[val].value_counts().max()

np.int64(1406)

In [63]:
df_train['SalePrice']

0       208500
1       181500
2       223500
3       140000
4       250000
         ...  
1455    175000
1456    210000
1457    266500
1458    142125
1459    147500
Name: SalePrice, Length: 1460, dtype: int64

In [59]:
int(df_train[val].value_counts().max()) < len(df_train) / 2

False

In [54]:
df_train = pd.read_csv(cnf.path['train_path'])

columns_with_na = df_train.columns[df_train.isna().any()].tolist()
for i, val in enumerate(columns_with_na):
    if pd.api.types.is_numeric_dtype(df_train[val]):
        df_train[val] = df_train[val].fillna(df_train[val].median())
    else:
        if df_train[val].value_counts().max() < len(df_train) / 2:
            df_train[val] = df_train[val].fillna('Not info')
        else:
            df_train[val] = df_train[val].fillna(df_train[val].mode()[0])        

In [56]:
[1, 2, 3] + ['str']

[1, 2, 3, 'str']

In [ ]:
df_train

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,Not info,Reg,Lvl,AllPub,...,0,Not info,Not info,Not info,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,Not info,Reg,Lvl,AllPub,...,0,Not info,Not info,Not info,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,Not info,IR1,Lvl,AllPub,...,0,Not info,Not info,Not info,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,Not info,IR1,Lvl,AllPub,...,0,Not info,Not info,Not info,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,Not info,IR1,Lvl,AllPub,...,0,Not info,Not info,Not info,0,12,2008,WD,Normal,250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,1456,60,RL,62.0,7917,Pave,Not info,Reg,Lvl,AllPub,...,0,Not info,Not info,Not info,0,8,2007,WD,Normal,175000
1456,1457,20,RL,85.0,13175,Pave,Not info,Reg,Lvl,AllPub,...,0,Not info,MnPrv,Not info,0,2,2010,WD,Normal,210000
1457,1458,70,RL,66.0,9042,Pave,Not info,Reg,Lvl,AllPub,...,0,Not info,GdPrv,Shed,2500,5,2010,WD,Normal,266500
1458,1459,20,RL,68.0,9717,Pave,Not info,Reg,Lvl,AllPub,...,0,Not info,Not info,Not info,0,4,2010,WD,Normal,142125


In [14]:
columns_with_na = df_test.columns[df_test.isna().any()].tolist()
filtered_df = df_test[columns_with_na]
filtered_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1459 entries, 0 to 1458
Data columns (total 33 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   MSZoning      1455 non-null   str    
 1   LotFrontage   1232 non-null   float64
 2   Alley         107 non-null    str    
 3   Utilities     1457 non-null   str    
 4   Exterior1st   1458 non-null   str    
 5   Exterior2nd   1458 non-null   str    
 6   MasVnrType    565 non-null    str    
 7   MasVnrArea    1444 non-null   float64
 8   BsmtQual      1415 non-null   str    
 9   BsmtCond      1414 non-null   str    
 10  BsmtExposure  1415 non-null   str    
 11  BsmtFinType1  1417 non-null   str    
 12  BsmtFinSF1    1458 non-null   float64
 13  BsmtFinType2  1417 non-null   str    
 14  BsmtFinSF2    1458 non-null   float64
 15  BsmtUnfSF     1458 non-null   float64
 16  TotalBsmtSF   1458 non-null   float64
 17  BsmtFullBath  1457 non-null   float64
 18  BsmtHalfBath  1457 non-null   float64
 

In [65]:
sub = pd.read_csv(r"C:\Users\Vsani\ML\House_Prices\results\inference3_Первый_запуск_inference_без_ДЛ\all_predictions.csv")
cols = ['LGBM_6', 'XGBoost_7', 'CatBoost_8']        # по val_RMSE из валидации
blend = np.expm1(np.log1p(sub[cols]).mean(axis=1))
pd.DataFrame({'Id': sub['Id'], 'SalePrice': blend}).to_csv('blend.csv', index=False)

In [73]:
df_train

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,Not info,Reg,Lvl,AllPub,...,0,Not info,Not info,Not info,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,Not info,Reg,Lvl,AllPub,...,0,Not info,Not info,Not info,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,Not info,IR1,Lvl,AllPub,...,0,Not info,Not info,Not info,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,Not info,IR1,Lvl,AllPub,...,0,Not info,Not info,Not info,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,Not info,IR1,Lvl,AllPub,...,0,Not info,Not info,Not info,0,12,2008,WD,Normal,250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,1456,60,RL,62.0,7917,Pave,Not info,Reg,Lvl,AllPub,...,0,Not info,Not info,Not info,0,8,2007,WD,Normal,175000
1456,1457,20,RL,85.0,13175,Pave,Not info,Reg,Lvl,AllPub,...,0,Not info,MnPrv,Not info,0,2,2010,WD,Normal,210000
1457,1458,70,RL,66.0,9042,Pave,Not info,Reg,Lvl,AllPub,...,0,Not info,GdPrv,Shed,2500,5,2010,WD,Normal,266500
1458,1459,20,RL,68.0,9717,Pave,Not info,Reg,Lvl,AllPub,...,0,Not info,Not info,Not info,0,4,2010,WD,Normal,142125


In [77]:
df_train = pd.read_csv(cnf.path['train_path'])
y = np.log1p(df_train['SalePrice'])
ensemble = pd.read_csv(r"C:\Users\Vsani\ML\House_Prices\results\ensemble_result_Запуск с ансамблем.csv")
cols = ['LGBM_5', 'XGBoost_6', 'CatBoost_7']
print(root_mean_squared_error(y, ensemble[cols].mean(axis=1)))     # RMSE ансамбля

0.12454263324180158


In [71]:
ensemble

,RandomForest_0,LinReg_1,KNN_2,KNN_3,DecisionTree_4,LGBM_5,XGBoost_6,CatBoost_7,nn_model_8
0,12.241016,12.235653,12.290214,12.277426,12.181523,12.198669,12.208543,12.231406,12.233156
1,12.040542,12.230355,12.078566,12.085737,12.061092,12.017851,12.076450,12.139434,12.147728
2,12.269279,12.254171,12.337656,12.322963,12.181523,12.232033,12.248211,12.260121,12.239061
3,12.123075,12.032895,11.995574,11.947599,12.132901,12.242702,12.205333,12.117338,12.209967
4,12.617539,12.567580,12.629288,12.660170,12.679236,12.734316,12.677427,12.671055,12.511414
...,...,...,...,...,...,...,...,...,...
1455,12.082423,12.112590,12.096089,12.096638,12.145168,12.084439,12.086373,12.062130,12.058660
1456,12.231520,12.207078,12.014211,11.954619,12.222222,12.238582,12.227942,12.162142,12.154693
1457,12.361991,12.600207,12.277165,12.308955,12.463019,12.482383,12.506310,12.414621,12.302061
1458,11.761846,11.852835,11.846096,11.820510,11.788418,11.837441,11.815042,11.771685,11.841576
